In [9]:
import pandas as pd
import joblib

import mlflow
import mlflow.sklearn


from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split

In [5]:
# Ml Flow experiment tracking
mlflow.set_experiment("Sepsis Survival Prediction")

<Experiment: artifact_location=('file:///c:/Users/laksh/Projects/Final project Sepsis Survival & Risk '
 'Prediction Platform/notebook/mlruns/1'), creation_time=1773557376602, experiment_id='1', last_update_time=1773557376602, lifecycle_stage='active', name='Sepsis Survival Prediction', tags={}, workspace='default'>

In [6]:
# Load processed data set
df = pd.read_csv("../data/sepsis_model_data.csv")

df.head()

,age,sex,episode,age_episode_interaction,outcome
0,21,1,1,21,1
1,20,1,1,20,1
2,21,1,1,21,1
3,77,0,1,77,1
4,72,0,1,72,1


In [7]:
# Define features and target variable
X = df.drop("outcome", axis=1)
y = df["outcome"]

In [10]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [13]:
logistic_model = joblib.load(r"C:\Users\laksh\Projects\Final project Sepsis Survival & Risk Prediction Platform\notebook\models\logistic_regression_model.pkl")

rf_model = joblib.load(r"C:\Users\laksh\Projects\Final project Sepsis Survival & Risk Prediction Platform\notebook\models\random_forest_model.pkl")

xgb_model = joblib.load(r"C:\Users\laksh\Projects\Final project Sepsis Survival & Risk Prediction Platform\notebook\models\xgboost_model.pkl")

cat_model = joblib.load(r"C:\Users\laksh\Projects\Final project Sepsis Survival & Risk Prediction Platform\notebook\models\catboost_model.pkl")

In [16]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "CatBoost": cat_model
}

In [20]:
for model_name, model in models.items():

    with mlflow.start_run(run_name=model_name):

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:,1]

        acc = accuracy_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        roc = roc_auc_score(y_test, y_prob)

        mlflow.log_param("model_name", model_name)

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("roc_auc", roc)

        mlflow.sklearn.log_model(
            model,
            "model",
            registered_model_name="SepsisSurvivalModel"
        )

        print(model_name, acc, recall, roc)

2026/03/15 18:32:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/15 18:32:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'SepsisSurvivalModel'.
Created version '1' of model 'SepsisSurvivalModel'.


Logistic Regression 0.5793748015062837 0.566405484818805 0.713975273120122


2026/03/15 18:32:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/15 18:32:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'SepsisSurvivalModel' already exists. Creating a new version of this model...
Created version '2' of model 'SepsisSurvivalModel'.


Random Forest 0.5593212649153849 0.5445151811949069 0.7035008196171576


2026/03/15 18:32:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/15 18:32:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'SepsisSurvivalModel' already exists. Creating a new version of this model...
Created version '3' of model 'SepsisSurvivalModel'.
2026/03/15 18:32:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/15 18:32:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code

XGBoost 0.5370899686947053 0.5176787463271303 0.7098740152056655
CatBoost 0.5390408783630507 0.5199314397649364 0.7103319494804057


Registered model 'SepsisSurvivalModel' already exists. Creating a new version of this model...
Created version '4' of model 'SepsisSurvivalModel'.
